# Практика · Двоетапні детектори> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.md](homework.md)> ⏱ Зошит навчає три мережі кандидат-областей (по одній на зерно) і кілька разів> проганяє ResNet-50 та ResNet-18. Заміряно: **від двох до трьох хвилин** на> процесорі в **одному потоці**, без відеокарти (два прогони дали 129 і 183 с —> різниця в тому, наскільки машина була зайнята іншим).Що зробимо:1. порахуємо, скільки вікон дає повний перебір і скільки це коштує;2. зміряємо ціну R-CNN — дві тисячі проходів мережі на один кадр;3. зміряємо Fast R-CNN — один прохід тіла плюс вибірка з карти ознак;4. покажемо на числах, як округлення координат у RoI-пулінгу зсуває ознаки;5. **напишемо RoI-Align руками** й звіримо з `torchvision.ops.roi_align`;6. **згенеруємо якорі руками** й звіримо кількість зі справжнім `AnchorGenerator`;7. зберемо спрощений RPN і виміряємо повноту кандидат-областей;8. розберемо `fasterrcnn_resnet50_fpn` по частинах і порівняємо з RetinaNet.

## 0 · Налаштування`torch.set_num_threads(1)` стоїть тут не для краси. На маленьких тензорах кількапотоків не пришвидшують нічого, зате додавання чисел із рухомою комою йде віншому порядку — і числа від прогону до прогону перестають збігатися.

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import roi_align, box_iou, nms

torch.set_num_threads(1)          # швидкість і, головне, відтворюваність
torch.manual_seed(0)

print("torch", torch.__version__)
import torchvision
print("torchvision", torchvision.__version__)
print("потоків:", torch.get_num_threads())

## 1 · Наскрізний приклад: сцени 64×64Той самий датасет, що в усьому блоці. На полотні 64×64 лежить від одного дотрьох предметів трьох класів. Рамка **не задається окремо**, а рахується змаски фігури — тому вона істинна за побудовою, і жодної розмітки нам не треба.

In [ ]:
SIZE = 64
CLASS_NAMES = ("коло", "квадрат", "трикутник")


def draw_one(canvas, rng):
    """Малює один предмет і повертає (клас, рамка). Рамка — з маски."""
    kind = int(rng.integers(0, 3))
    radius = int(rng.integers(6, 11))
    cx = int(rng.integers(radius + 1, SIZE - radius - 1))
    cy = int(rng.integers(radius + 1, SIZE - radius - 1))

    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    dx = xs - cx
    dy = ys - cy
    if kind == 0:
        mask = dx * dx + dy * dy <= radius * radius
    elif kind == 1:
        mask = (np.abs(dx) <= radius) & (np.abs(dy) <= radius)
    else:
        # трикутник: вершина вгорі, основа внизу
        t = (dy + radius) / (2 * radius)
        mask = (dy >= -radius) & (dy <= radius) & (np.abs(dx) <= radius * np.clip(t, 0, 1))

    canvas[mask] = 1.0
    rows, cols = np.nonzero(mask)
    box = (float(cols.min()), float(rows.min()), float(cols.max() + 1), float(rows.max() + 1))
    return kind, box


def make_scene(rng):
    canvas = np.zeros((SIZE, SIZE), dtype=np.float32)
    labels, boxes = [], []
    for _ in range(int(rng.integers(1, 4))):
        kind, box = draw_one(canvas, rng)
        labels.append(kind)
        boxes.append(box)
    canvas = canvas + rng.normal(0, 0.12, canvas.shape).astype(np.float32)
    return np.clip(canvas, 0, 1), np.array(boxes, dtype=np.float32), np.array(labels)


def make_dataset(n_scenes, seed):
    rng = np.random.default_rng(seed)
    images, all_boxes, all_labels = [], [], []
    for _ in range(n_scenes):
        image, boxes, labels = make_scene(rng)
        images.append(image)
        all_boxes.append(boxes)
        all_labels.append(labels)
    return np.stack(images), all_boxes, all_labels


train_images, train_boxes, train_labels = make_dataset(400, seed=42)
test_images, test_boxes, test_labels = make_dataset(120, seed=7)

print(f"навчальних сцен {len(train_images)}, у них рамок {sum(len(b) for b in train_boxes)}")
print(f"перевірочних сцен {len(test_images)}, у них рамок {sum(len(b) for b in test_boxes)}")

sides = np.concatenate([np.stack([b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]], 1).ravel()
                        for b in train_boxes])
print(f"сторона рамки: від {sides.min():.0f} до {sides.max():.0f} px, середня {sides.mean():.2f}")

Подивимось на одну сцену з її істинними рамками. Це та сама сцена, щонамальована в лекції в інтерактиві про якорі.

In [ ]:
import matplotlib.pyplot as plt

demo_images, demo_boxes, demo_labels = make_dataset(36, seed=42)
scene, boxes, labels = demo_images[35], demo_boxes[35], demo_labels[35]

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(scene, cmap="gray", interpolation="nearest")
for box, label in zip(boxes, labels):
    x1, y1, x2, y2 = box
    ax.add_patch(plt.Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1,
                               fill=False, edgecolor="red", linewidth=1.5))
    ax.text(x1 - 0.5, y1 - 1.5, CLASS_NAMES[label], color="red", fontsize=9)
ax.set_title("сцена 64×64 з трьома предметами")
ax.axis("off")
plt.show()

print("рамки цієї сцени:", boxes.tolist())

## 2 · Скільки вікон дає повний перебірНайпростіший спосіб знайти предмет — перебрати всі прямокутники. Порахуємо, ускільки положень стає вікно на фотографії 1000×600 при різних кроках. Формулаодна: скільки разів вікно поміститься по горизонталі, помножити на скільки разівпоміститься по вертикалі.

In [ ]:
FRAME_W, FRAME_H = 1000, 600
WINDOW_SCALES = [32, 48, 64, 96, 128, 192, 256, 384, 512]
ASPECT_RATIOS = [0.5, 1.0, 2.0]


def window_count(scale, ratio, step):
    """Скільки положень має одне вікно заданого розміру й форми."""
    width = round(scale / math.sqrt(ratio))
    height = round(scale * math.sqrt(ratio))
    if width > FRAME_W or height > FRAME_H:
        return 0
    along_x = (FRAME_W - width) // step + 1
    along_y = (FRAME_H - height) // step + 1
    return along_x * along_y


def total_windows(step):
    return sum(window_count(s, r, step) for s in WINDOW_SCALES for r in ASPECT_RATIOS)


print("одне вікно 128×128 при кроці 4:", window_count(128, 1.0, 4), "положень")
print()
print("крок | усього вікон на кадр")
for step in (1, 2, 4, 8, 16):
    print(f"{step:>4} | {total_windows(step):>12,}".replace(",", " "))

## 3 · Замір 1: ціна R-CNNR-CNN бере близько двох тисяч кандидат-областей від Selective Search і проганяє**кожну окремо** крізь згорткову мережу. Зміряємо один такий прохід і помножимо.> Абсолютні секунди залежать від твоєї машини й від того, чим вона зайнята, тому> в тебе вийдуть інші числа, ніж у лекції. Незмінне тут **співвідношення**:> скільки разів працює тіло мережі.

In [ ]:
from torchvision.models import resnet18

classifier = resnet18(weights=None).eval()       # архітектура без завантаження ваг
crop = torch.randn(1, 3, 224, 224)               # один вирізаний і розтягнутий шматок

with torch.no_grad():
    for _ in range(3):                            # прогріваємо, щоб не міряти перший виклик
        classifier(crop)
    times = []
    for _ in range(10):
        start = time.perf_counter()
        classifier(crop)
        times.append(time.perf_counter() - start)

times.sort()
one_crop = times[len(times) // 2]                 # медіана стійкіша за середнє
print(f"один прохід ResNet-18 по шматку 224×224: {one_crop*1000:.1f} мс")

N_PROPOSALS = 2000
per_image = one_crop * N_PROPOSALS
print(f"R-CNN на одне зображення: {N_PROPOSALS} проходів = {per_image:.1f} с")
print(f"R-CNN на 5000 зображень: {N_PROPOSALS*5000:,} проходів = {per_image*5000/3600:.0f} год"
      .replace(",", " "))
print(f"тобто {per_image*5000/3600/24:.1f} доби безперервної роботи на один прохід датасету")

## 4 · Замір 2: ціна Fast R-CNNFast R-CNN міняє порядок дій: спершу один прохід тіла по **цілому** кадру, потімвибірка ділянок із готової карти ознак. Зміряємо обидві частини окремо, щоб буловидно, з чого складається час.

In [ ]:
# тіло ResNet-18 до четвертого блоку: сумарний крок 16
backbone = nn.Sequential(*list(resnet18(weights=None).children())[:-3]).eval()
frame = torch.randn(1, 3, 600, 1000)

with torch.no_grad():
    feature_map = backbone(frame)
print("карта ознак цілого кадру:", tuple(feature_map.shape))

with torch.no_grad():
    for _ in range(2):
        backbone(frame)
    times = []
    for _ in range(5):
        start = time.perf_counter()
        backbone(frame)
        times.append(time.perf_counter() - start)
times.sort()
body_once = times[len(times) // 2]
print(f"один прохід тіла по кадру 600×1000: {body_once*1000:.0f} мс")

Тепер друга частина: вибрати з карти ознак 2 000 ділянок і прогнати їх крізьголову. Голова тут така сама, як у справжньому Fast R-CNN: два повнозвʼязні шарипо 1 024 нейрони.

In [ ]:
rng = np.random.default_rng(0)
proposal_boxes = torch.tensor(rng.uniform(0, 400, size=(2000, 4)), dtype=torch.float32)
proposal_boxes[:, 2:] += proposal_boxes[:, :2] + 30          # робимо рамки додатної площі
rois = torch.cat([torch.zeros(2000, 1), proposal_boxes], dim=1)   # [номер кадру, x1,y1,x2,y2]

head = nn.Sequential(nn.Flatten(),
                     nn.Linear(256 * 7 * 7, 1024), nn.ReLU(),
                     nn.Linear(1024, 1024), nn.ReLU(),
                     nn.Linear(1024, 91)).eval()


def head_time(n_boxes):
    """Час на вибірку n рамок із готової карти ознак плюс прохід голови."""
    part = rois[:n_boxes]
    with torch.no_grad():
        for _ in range(2):
            head(roi_align(feature_map, part, (7, 7), spatial_scale=1/16, aligned=True))
        measured = []
        for _ in range(5):
            start = time.perf_counter()
            pooled = roi_align(feature_map, part, (7, 7), spatial_scale=1/16, aligned=True)
            head(pooled)
            measured.append(time.perf_counter() - start)
    measured.sort()
    return measured[len(measured) // 2]


print("рамок | голова, мс | разом із тілом, мс")
for n in (100, 300, 2000):
    t = head_time(n)
    print(f"{n:>5} | {t*1000:>10.0f} | {(body_once+t)*1000:>18.0f}")

fast_2000 = body_once + head_time(2000)
print()
print(f"R-CNN:      {per_image*1000:>9.0f} мс на кадр")
print(f"Fast R-CNN: {fast_2000*1000:>9.0f} мс на кадр")
print(f"виграш Fast R-CNN: {per_image/fast_2000:.0f}×")

## 5 · Замір 3: округлення проти білінійної вибіркиГоловна технічна деталь другого етапу. Щоб побачити зсув **числом**, а не наоко, візьмемо особливу карту ознак: значення клітинки дорівнює її горизонтальнійкоординаті. Тоді середнє значення вирізаного вікна — це просто координата йогоцентра, і зсув видно прямо у відповіді.Одна тонкість: округлення тут має бути **таке саме, як у C і в JavaScript** —половина йде вгору. Вбудований `round` у Python округлює 2.5 до 2 («допарного»), і числа розійшлися б із лекцією.

In [ ]:
MAP_SIDE = 12
STRIDE = 16
OUT = 7

# карта-пандус: f[рядок, стовпчик] = стовпчик
ramp = torch.arange(MAP_SIDE, dtype=torch.float32).repeat(MAP_SIDE, 1)[None, None]


def round_half_up(value):
    """Округлення, як у C: половина йде вгору. Вбудований round() округлює до парного."""
    return int(math.floor(value + 0.5))


def quantised_centre(x1, x2):
    """RoI-пулінг: округлюємо межі рамки, потім межі комірок. Два округлення підряд."""
    q1 = round_half_up(x1 / STRIDE)
    q2 = round_half_up(x2 / STRIDE)
    cell = (q2 - q1) / OUT
    total = 0.0
    for j in range(OUT):
        left = q1 + int(j * cell)
        right = q1 + int((j + 1) * cell)
        if right <= left:
            right = left + 1
        total += sum(range(left, right)) / (right - left)     # середнє по комірці
    return total / OUT


print("зсув | центр істинний | RoI-пулінг | RoI-Align | помилка пулінгу, px")
errors = []
for shift in range(16):
    x1 = 36.0 + shift
    x2 = x1 + 88.0
    true_centre = (x1 + x2) / 2 / STRIDE - 0.5          # у координатах центрів клітинок
    quantised = quantised_centre(x1, x2)
    box = torch.tensor([[0.0, x1, 32.0, x2, 112.0]])
    aligned = roi_align(ramp, box, (OUT, OUT), spatial_scale=1/STRIDE,
                        sampling_ratio=2, aligned=True).mean().item()
    error = (quantised - true_centre) * STRIDE
    errors.append(error)
    print(f"{shift:>4} | {true_centre:>14.4f} | {quantised:>10.4f} | {aligned:>9.4f} | {error:>19.2f}")

absolute = [abs(e) for e in errors]
print()
print(f"модуль зсуву RoI-пулінгу: від {min(absolute):.2f} до {max(absolute):.2f} px, "
      f"у середньому {sum(absolute)/len(absolute):.2f}")
print("RoI-Align у всіх шістнадцяти положеннях збігається з істиною до останнього знака")

А тепер про аргумент `aligned`. Центр нульового пікселя лежить у координаті 0.5,а не 0. Якщо цього не врахувати, уся ділянка їде рівно на пів клітинки — прикроці 16 це вісім пікселів, стабільно, у будь-якому положенні рамки.

In [ ]:
box = torch.tensor([[0.0, 36.0, 32.0, 124.0, 112.0]])
true_centre = (36.0 + 124.0) / 2 / STRIDE - 0.5

for flag in (True, False):
    got = roi_align(ramp, box, (OUT, OUT), spatial_scale=1/STRIDE,
                    sampling_ratio=2, aligned=flag).mean().item()
    print(f"aligned={str(flag):<5} центр {got:.4f}, зсув {(got-true_centre)*STRIDE:+.2f} px")

print()
print("замовчування torchvision — aligned=False, тобто зі зсувом;")
print("у новому коді треба ставити aligned=True свідомо")

## 6 · Пишемо RoI-Align рукамиНайкорисніша перевірка практики: всередині бібліотеки немає магії. Напишемобілінійну вибірку самі — чотири сусіди, ваги з дробових частин — і звіриморезультат із `torchvision.ops.roi_align`.

In [ ]:
def bilinear(feature, y, x):
    """Значення карти ознак у дробовій точці: зважене середнє чотирьох сусідів."""
    height, width = feature.shape
    if y < -1.0 or x < -1.0 or y > height or x > width:
        return 0.0                                   # точка поза картою — нуль
    y = max(y, 0.0)
    x = max(x, 0.0)
    y0, x0 = int(y), int(x)
    y1, x1 = y0 + 1, x0 + 1
    if y0 >= height - 1:                             # притискаємось до останнього рядка
        y1 = y0 = height - 1
        y = float(y0)
    if x0 >= width - 1:
        x1 = x0 = width - 1
        x = float(x0)
    dy = y - y0
    dx = x - x0
    return ((1 - dy) * (1 - dx) * feature[y0, x0] + (1 - dy) * dx * feature[y0, x1]
            + dy * (1 - dx) * feature[y1, x0] + dy * dx * feature[y1, x1])


def my_roi_align(feature, box, out_size, spatial_scale, samples=2):
    """RoI-Align руками: жодного округлення координат."""
    out_h, out_w = out_size
    # aligned=True: пів пікселя за те, що центр нульового пікселя лежить у 0.5
    x1 = box[0] * spatial_scale - 0.5
    y1 = box[1] * spatial_scale - 0.5
    x2 = box[2] * spatial_scale - 0.5
    y2 = box[3] * spatial_scale - 0.5
    cell_w = (x2 - x1) / out_w
    cell_h = (y2 - y1) / out_h
    result = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            total = 0.0
            for iy in range(samples):
                sample_y = y1 + i * cell_h + (iy + 0.5) * cell_h / samples
                for ix in range(samples):
                    sample_x = x1 + j * cell_w + (ix + 0.5) * cell_w / samples
                    total += bilinear(feature, sample_y, sample_x)
            result[i, j] = total / (samples * samples)
    return result


check_rng = np.random.default_rng(42)
random_map = check_rng.normal(size=(20, 20)).astype(np.float32)
map_tensor = torch.tensor(random_map)[None, None]

for coords in ([100.3, 40.7, 187.7, 121.9], [12.0, 8.0, 60.0, 56.0], [5.5, 5.5, 300.5, 300.5]):
    library = roi_align(map_tensor, torch.tensor([[0.0] + coords]), (7, 7),
                        spatial_scale=1/16, sampling_ratio=2, aligned=True)[0, 0].numpy()
    mine = my_roi_align(random_map, coords, (7, 7), 1/16)
    assert np.allclose(library, mine, atol=1e-5), "розрахунок розійшовся!"
    print(f"рамка {coords}: максимальна різниця {np.abs(library-mine).max():.2e} ✅ збігається")

## 7 · Якорі рукамиТепер згенеруємо сітку якорів самі: центр кожної клітинки карти ознак × кількамасштабів × кілька співвідношень сторін. І одразу звіримо кількість зі справжнімгенератором із `torchvision`.

In [ ]:
GRID = 16          # карта ознак 16×16 для сцени 64×64
ANCHOR_STRIDE = 4  # одна клітинка карти = 4 пікселі зображення
SCALES = (8, 12, 18)
RATIOS = (0.5, 1.0, 2.0)


def make_anchors(scales=SCALES, ratios=RATIOS, grid=GRID, stride=ANCHOR_STRIDE):
    """Сітка якорів: центр клітинки × масштаб × співвідношення сторін."""
    boxes = []
    for row in range(grid):
        for col in range(grid):
            cy = (row + 0.5) * stride
            cx = (col + 0.5) * stride
            for scale in scales:
                for ratio in ratios:
                    # ratio = висота / ширина при сталій площі scale²
                    height = scale * math.sqrt(ratio)
                    width = scale / math.sqrt(ratio)
                    boxes.append([cx - width / 2, cy - height / 2,
                                  cx + width / 2, cy + height / 2])
    return torch.tensor(boxes, dtype=torch.float32)


anchors = make_anchors()
print(f"якорів на нашу сцену: {anchors.shape[0]} = {GRID}·{GRID}·{len(SCALES)}·{len(RATIOS)}")
print("перші три якорі:", [[round(v, 2) for v in a] for a in anchors[:3].tolist()])

Скільки якорів у справжній моделі? Порахуємо формулою й одразу перевіримосправжнім `AnchorGenerator`, підсунувши йому карти ознак потрібних розмірів.Ваг ми нікуди не тягнемо: `weights=None, weights_backbone=None` будує саму лишеархітектуру.

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, retinanet_resnet50_fpn
from torchvision.models.detection.image_list import ImageList

detector = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None).eval()

FRAME = (800, 1216)
LEVELS = [(200, 304), (100, 152), (50, 76), (25, 38), (13, 19)]

by_hand = sum(h * w for h, w in LEVELS) * 3      # три співвідношення сторін на клітинку
print("наша формула:", f"{by_hand:,}".replace(",", " "), "якорів")

fake_maps = [torch.zeros(1, 256, h, w) for h, w in LEVELS]
images = ImageList(torch.zeros(1, 3, *FRAME), [FRAME])
real = detector.rpn.anchor_generator(images, fake_maps)[0]
print("AnchorGenerator:", f"{real.shape[0]:,}".replace(",", " "), "якорів")

assert by_hand == real.shape[0], "кількість якорів розійшлася!"
print("✅ збігається")

print()
print("рівень | карта    | клітинок | якорів")
for (h, w), name in zip(LEVELS, ["P2", "P3", "P4", "P5", "P6"]):
    shape = f"{h}×{w}"
    print(f"{name:>6} | {shape:<8} | {h*w:>8,} | {h*w*3:>7,}".replace(",", " "))

## 8 · Замір 4: скільки якорів позитивніЯкір вважається позитивним, якщо його IoU з якоюсь істинною рамкою більший за0.7. Порахуємо, скільки таких на 200 сценах. Заразом перевіримо, скільки істиннихрамок узагалі не мають підхожого якоря — саме через це в Faster R-CNN є рятівнеправило.

In [ ]:
POSITIVE_IOU = 0.7

sample_images, sample_boxes, _ = make_dataset(200, seed=42)

total_anchors = 0
positive_anchors = 0
positive_at_05 = 0
total_truth = 0
truth_without_anchor = 0
best_per_truth = []

for boxes in sample_boxes:
    truth = torch.tensor(boxes)
    overlap = box_iou(anchors, truth)            # [якорі, істини]
    best_for_anchor = overlap.max(dim=1).values
    total_anchors += anchors.shape[0]
    positive_anchors += int((best_for_anchor > POSITIVE_IOU).sum())
    positive_at_05 += int((best_for_anchor > 0.5).sum())
    best_for_truth = overlap.max(dim=0).values
    total_truth += truth.shape[0]
    truth_without_anchor += int((best_for_truth <= POSITIVE_IOU).sum())
    best_per_truth.append(best_for_truth.numpy())

share = 100 * positive_anchors / total_anchors
print(f"якорів усього: {total_anchors:,}".replace(",", " "))
print(f"позитивних (IoU > {POSITIVE_IOU}): {positive_anchors} — {share:.3f} %")
print(f"при мʼякшому порозі 0.5: {positive_at_05} — {100*positive_at_05/total_anchors:.3f} %")
print(f"на один якір-предмет припадає {(total_anchors-positive_anchors)//positive_anchors} "
      f"якорів-фону")
print()
print(f"істинних рамок: {total_truth}")
print(f"з них без жодного якоря IoU > {POSITIVE_IOU}: {truth_without_anchor} "
      f"({100*truth_without_anchor/total_truth:.1f} %)")
best_per_truth = np.concatenate(best_per_truth)
print(f"найкращий IoU на істину: середній {best_per_truth.mean():.3f}, "
      f"мінімум {best_per_truth.min():.3f}")

Тепер подивимось, які саме з девʼяти варіантів якоря взагалі колисьспрацьовують. Наші предмети — коло, квадрат і трикутник, і рамка в усіх майжеквадратна. Здогадка очевидна; перевіримо її числом.

In [ ]:
print("масштаб | співвідношення | позитивних на 200 сценах")
for scale in SCALES:
    for ratio in RATIOS:
        subset = make_anchors(scales=(scale,), ratios=(ratio,))
        count = 0
        for boxes in sample_boxes:
            overlap = box_iou(subset, torch.tensor(boxes))
            count += int((overlap.max(dim=1).values > POSITIVE_IOU).sum())
        print(f"{scale:>7} | {ratio:>14} | {count:>24}")

print()
print("сім варіантів із девʼяти не дали жодного позитивного якоря —")
print("набір якорів має відповідати формі предметів, а не братися зі статті")

## 9 · Замір 5: спрощений RPNЗберемо мережу кандидат-областей. Будова рівно та, що в лекції: згорткове тіло робитьіз сцени 64×64 карту 16×16, спільна згортка 3×3 готує ознаки, а дві згортки 1×1видають для кожного з девʼяти якорів клітинки впевненість («предмет чи фон») ічотири числа зсуву.

In [ ]:
N_ANCHORS_PER_CELL = len(SCALES) * len(RATIOS)

ANCHOR_W = anchors[:, 2] - anchors[:, 0]
ANCHOR_H = anchors[:, 3] - anchors[:, 1]
ANCHOR_CX = anchors[:, 0] + ANCHOR_W / 2
ANCHOR_CY = anchors[:, 1] + ANCHOR_H / 2


def encode(truth):
    """Істинна рамка → зсув відносно якоря: (наскільки посунути, у скільки разів змінити)."""
    tw = truth[:, 2] - truth[:, 0]
    th = truth[:, 3] - truth[:, 1]
    tcx = truth[:, 0] + tw / 2
    tcy = truth[:, 1] + th / 2
    return torch.stack([(tcx - ANCHOR_CX) / ANCHOR_W,
                        (tcy - ANCHOR_CY) / ANCHOR_H,
                        torch.log(tw / ANCHOR_W),
                        torch.log(th / ANCHOR_H)], dim=1)


def decode(offsets):
    """Зсув → рамка в пікселях."""
    cx = ANCHOR_CX + offsets[:, 0] * ANCHOR_W
    cy = ANCHOR_CY + offsets[:, 1] * ANCHOR_H
    w = ANCHOR_W * torch.exp(offsets[:, 2].clamp(-2, 2))    # clamp рятує від exp(великого)
    h = ANCHOR_H * torch.exp(offsets[:, 3].clamp(-2, 2))
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=1)


class RPN(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
        )                                                   # 64×64 → 16×16, крок 4
        self.shared = nn.Sequential(nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
        self.objectness = nn.Conv2d(32, N_ANCHORS_PER_CELL, 1)
        self.regression = nn.Conv2d(32, N_ANCHORS_PER_CELL * 4, 1)

    def forward(self, x):
        features = self.shared(self.body(x))
        n = x.shape[0]
        # переставляємо осі так, щоб порядок збігся з порядком генерації якорів
        scores = self.objectness(features).permute(0, 2, 3, 1).reshape(n, -1)
        offsets = self.regression(features).permute(0, 2, 3, 1).reshape(n, -1, 4)
        return scores, offsets


print("ваг у RPN:", sum(p.numel() for p in RPN().parameters()))

Зіставлення якорів з істиною — окремий крок, і він не залежить від ваг, томурахуємо його один раз перед навчанням. Правило те саме, що в статті: більше0.7 — предмет, менше 0.3 — фон, між ними — ігноруємо. Плюс рятівне правило:найкращий якір кожної істини позитивний завжди.

In [ ]:
def match_anchors(truth):
    """Мітка для кожного якоря: 1 предмет, 0 фон, -1 ігнорувати."""
    overlap = box_iou(anchors, truth)
    best = overlap.max(dim=1)
    labels = torch.full((anchors.shape[0],), -1.0)
    labels[best.values < 0.3] = 0.0
    labels[best.values > 0.7] = 1.0
    labels[overlap.argmax(dim=0)] = 1.0            # рятівне правило
    return labels, truth[best.indices]


matched = [match_anchors(torch.tensor(b)) for b in train_boxes]
positives = sum(int((labels == 1).sum()) for labels, _ in matched)
print(f"позитивних якорів на 400 навчальних сценах: {positives} "
      f"(разом із тими, що їх додало рятівне правило)")
print(f"це {100*positives/(400*anchors.shape[0]):.3f} % усіх якорів —")
print("тому втрату рахуємо не по всіх, а по збалансованій вибірці")

In [ ]:
def train_rpn(seed, epochs=14):
    """Навчання RPN: втрата рахується по вибірці «усі позитивні + утричі більше фону»."""
    torch.manual_seed(seed)
    net = RPN()
    optimiser = torch.optim.AdamW(net.parameters(), lr=3e-3)
    inputs = torch.tensor(train_images)[:, None]
    order = np.arange(len(train_images))
    shuffler = np.random.default_rng(seed)

    for _ in range(epochs):
        shuffler.shuffle(order)
        for start in range(0, len(order), 16):
            batch = order[start:start + 16]
            scores, offsets = net(inputs[batch])
            loss = 0.0
            for position, index in enumerate(batch):
                labels, targets = matched[index]
                positive_idx = torch.nonzero(labels == 1).squeeze(1)
                negative_idx = torch.nonzero(labels == 0).squeeze(1)
                n_positive = len(positive_idx)
                take = max(n_positive * 3, 32)
                chosen_negative = negative_idx[torch.randperm(len(negative_idx))[:take]]
                selected = torch.cat([positive_idx, chosen_negative])
                answer = torch.zeros(len(selected))
                answer[:n_positive] = 1.0
                loss = loss + F.binary_cross_entropy_with_logits(scores[position][selected], answer)
                if n_positive:
                    loss = loss + F.smooth_l1_loss(offsets[position][positive_idx],
                                                   encode(targets)[positive_idx])
            loss = loss / len(batch)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
    return net.eval()


trained = []
for seed in (0, 1, 2):
    start = time.perf_counter()
    trained.append(train_rpn(seed))
    print(f"зерно {seed}: навчання {time.perf_counter()-start:.1f} с")

Тепер головний замір теми. Беремо перші **k** кандидат-областей за впевненістю (післяпритлумлення немаксимумів) і рахуємо **повноту**: яка частка істинних рамок маєсеред цих k хоча б одну з достатнім IoU. Точність генератора нас не цікавить —зайве відсіє другий етап, а втрачений предмет не повернути.

In [ ]:
K_VALUES = (1, 3, 5, 10, 20, 50, 100, 300)


@torch.no_grad()
def all_proposals(net, image):
    """Кандидат-області однієї сцени, відсортовані за впевненістю й прорідені NMS."""
    scores, offsets = net(torch.tensor(image)[None, None])
    confidence = torch.sigmoid(scores[0])
    boxes = decode(offsets[0]).clamp(0, SIZE)
    keep = nms(boxes, confidence, 0.7)
    return boxes[keep]


def recall_curve(net, iou_threshold):
    ready = [(all_proposals(net, image), torch.tensor(boxes))
             for image, boxes in zip(test_images, test_boxes)]
    curve = {}
    for k in K_VALUES:
        hit = total = 0
        for proposals, truth in ready:
            top = proposals[:k]
            total += len(truth)
            if len(top):
                hit += int((box_iou(truth, top).max(dim=1).values >= iou_threshold).sum())
        curve[k] = hit / total
    return curve


for iou_threshold in (0.5, 0.7, 0.75):
    rows = {k: [] for k in K_VALUES}
    for net in trained:
        curve = recall_curve(net, iou_threshold)
        for k in K_VALUES:
            rows[k].append(curve[k])
    print(f"── повнота кандидат-областей при IoU ≥ {iou_threshold} ──")
    for k in K_VALUES:
        values = np.array(rows[k])
        print(f"  кандидат-областей {k:>3}: {values.mean():.3f} ± {values.std():.3f}")
    print()

Число саме по собі мало що каже. Порівняймо навчений RPN із тим самим наборомякорів, узятим **навмання**, без жодного навчання. Саме ця різниця й пояснює,навіщо кандидат-області вчити.

In [ ]:
def random_recall(k, iou_threshold, repeats=3):
    """Повнота k якорів, узятих навмання. Зерно залежить лише від k і номера спроби,
    тому число відтворюється незалежно від порядку викликів."""
    results = []
    for attempt in range(repeats):
        picker = np.random.default_rng(1000 * attempt + k)
        hit = total = 0
        for boxes in test_boxes:
            index = picker.choice(anchors.shape[0], size=min(k, anchors.shape[0]),
                                  replace=False)
            overlap = box_iou(torch.tensor(boxes), anchors[index])
            hit += int((overlap.max(dim=1).values >= iou_threshold).sum())
            total += len(boxes)
        results.append(hit / total)
    return float(np.mean(results))


curves = [recall_curve(net, 0.5) for net in trained]
print("кандидат-областей | навчений RPN | навмання")
for k in (10, 50, 100, 300, 1000, 2304):
    if k in K_VALUES:
        trained_value = f"{np.mean([c[k] for c in curves]):.3f}"
    else:
        trained_value = "   —"
    print(f"{k:>10} | {trained_value:>12} | {random_recall(k, 0.5):.3f}")

print()
print("навченому RPN вистачає десяти кандидат-областей там, де випадковим якорям")
print("треба тисяча — у сто разів менше кандидатів при тій самій повноті")

In [ ]:
# скільки коштує один прогін RPN на сцені
net = trained[0]
with torch.no_grad():
    for _ in range(5):
        all_proposals(net, test_images[0])
    measured = []
    for i in range(50):
        start = time.perf_counter()
        all_proposals(net, test_images[i % len(test_images)])
        measured.append(time.perf_counter() - start)
measured.sort()
print(f"один прогін RPN на сцені 64×64: {measured[len(measured)//2]*1000:.2f} мс")

## 10 · Замір 6: розбір справжньої моделіПорахуємо, куди йдуть 41 808 406 ваг `fasterrcnn_resnet50_fpn`, і порівняємо зодноетапною `retinanet_resnet50_fpn`.

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())


total = count(detector)
print(f"fasterrcnn_resnet50_fpn: {total:,} ваг".replace(",", " "))
print()
parts = [
    ("хребет: тіло ResNet-50", detector.backbone.body),
    ("хребет: FPN", detector.backbone.fpn),
    ("RPN", detector.rpn),
    ("голова: fc6 і fc7", detector.roi_heads.box_head),
    ("голова: клас і рамка", detector.roi_heads.box_predictor),
]
for name, module in parts:
    weights = count(module)
    print(f"{name:<24} {weights:>12,}  {100*weights/total:>6.2f} %".replace(",", " "))

print()
print("усередині RPN:")
print(f"  згортка 3×3        {count(detector.rpn.head.conv):>10,}".replace(",", " "))
print(f"  предмет чи фон     {count(detector.rpn.head.cls_logits):>10,}".replace(",", " "))
print(f"  зсув рамки         {count(detector.rpn.head.bbox_pred):>10,}".replace(",", " "))
print()
print(f"другий етап дорожчий за RPN у "
      f"{count(detector.roi_heads)/count(detector.rpn):.0f} рази")

In [ ]:
one_stage = retinanet_resnet50_fpn(weights=None, weights_backbone=None).eval()
print(f"retinanet_resnet50_fpn: {count(one_stage):,} ваг".replace(",", " "))
print(f"  хребет: тіло   {count(one_stage.backbone.body):>12,}".replace(",", " "))
print(f"  хребет: FPN    {count(one_stage.backbone.fpn):>12,}".replace(",", " "))
print(f"  голова класів  {count(one_stage.head.classification_head):>12,}".replace(",", " "))
print(f"  голова рамок   {count(one_stage.head.regression_head):>12,}".replace(",", " "))
print()
print(f"RetinaNet легша на {count(detector)-count(one_stage):,} ваг".replace(",", " "))

Але це при 91 класі COCO. Голова Faster R-CNN працює по кількох сотняхкандидат-областей, а голова RetinaNet — по всіх якорях кожної точки згорткою 3×3, томукожен новий клас коштує їй набагато дорожче. Перевіримо, як це виглядає прирізній кількості класів.

In [ ]:
print("класів | Faster R-CNN | RetinaNet | хто легша")
for n_classes in (2, 91, 500, 587):
    a = count(fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None,
                                      num_classes=n_classes))
    b = count(retinanet_resnet50_fpn(weights=None, weights_backbone=None,
                                     num_classes=n_classes))
    lighter = "RetinaNet" if b < a else "Faster R-CNN"
    print(f"{n_classes:>6} | {a:>12,} | {b:>9,} | {lighter}".replace(",", " "))

print()
print("ціна одного класу: Faster R-CNN 5 125 ваг, RetinaNet 20 745 —")
print("рівно на 587 класах перевага одноетапної моделі за вагою зникає")

## Завдання### 🟢 Рівень 1Постав у `window_count` крок 2 замість 4 і порахуй, у скільки разів зрослакількість вікон. Поясни словами, чому дрібніший крок потрібен, хоч і дорожчий.### 🟡 Рівень 2Заміни в `make_anchors` набір `SCALES` і `RATIOS` так, щоб частка позитивнихякорів на 200 сценах зросла щонайменше вдвічі проти 0.079 %. Покажи число дой після і поясни, чому саме твій набір кращий **для цих даних**.### 🔴 Рівень 3Виміряй, як повнота кандидат-областей залежить від порога притлумлення немаксимумів у`all_proposals` (спробуй 0.3, 0.5, 0.7, 0.9). Побудуй таблицю «поріг × кількістькандидат-областей → повнота» і знайди поріг, при якому десяти кандидат-областей ще вистачає.